# ECG Signal Processing Workshop — Notebook 1: Data Parsing & Signal Overview

## Workshop Introduction

This workshop explores ECG signal processing through a comparative lens: we simultaneously record cardiac signals using two fundamentally different acquisition systems and then walk through the full analysis pipeline from raw data to clinical-grade peak detection.

### The Two Recording Systems

| Property | BIOPAC MP160 | Baby Belt (Textile Sensor) |
|---|---|---|
| **Electrode type** | Gelled Ag/AgCl adhesive | Dry textile (knitted conductive fabric) |
| **Digitiser** | AcqKnowledge via USB | On-body BLE module |
| **Sampling rate** | 1 000 Hz (crystal-locked) | ~100 Hz (nominal, Bluetooth-jittered) |
| **Resolution** | 16-bit, ±10 V | Raw ADC counts (large integers) |
| **Channels** | 11 (ECG, PPG, filtered ECG, HR, R-R, etc.) | ECG + Resp + SpO2 + IMU (accel/gyro/mag) |
| **Use case** | Gold-standard clinical reference | Wearable / neonatal monitoring |

By comparing these two systems on the same subject at the same time, we can rigorously evaluate how electrode type, sampling rate, and transmission jitter affect signal quality and downstream analysis.

### Workshop Overview

| Notebook | Title | Focus |
|---|---|---|
| **NB01** | Data Parsing & Signal Overview | File I/O, raw visualisation, PSD comparison |
| **NB02** | Classical Filtering & R-Peak Detection | Bandpass, Pan-Tompkins, NeuroKit2 |
| **NB03** | Wavelet & Time-Frequency Analysis | CWT, STFT, synchrosqueezing |
| **NB04** | Deep-Learning R-Peak Detection | RPNet, data augmentation, evaluation |
| **NB05** | HRV Feature Extraction | Time-, frequency-, and nonlinear-domain HRV |
| **NB06** | Cross-System Comparison & Reporting | Bland-Altman, concordance, summary |

This first notebook covers **data parsing** (reading two very different file formats into clean DataFrames), **signal visualisation** (raw waveforms, synchronised overlays), and a **power spectral density** comparison that reveals the frequency content of each recording chain.

## 1. Package Requirements

The cell below lists every Python package used across all six notebooks. Uncomment the `pip install` lines if you need to install them into your current environment.

In [ ]:
# =============================================================
# Uncomment the lines below to install all workshop dependencies
# =============================================================
%pip install numpy pandas scipy matplotlib plotly
%pip install neurokit2 wfdb ssqueezepy PyWavelets
%pip install tqdm torch torchvision seaborn
#consensus-peaks # Need to install manually https://github.com/LaussenLabs/consensus_peaks

## 2. File Path Configuration

Edit the three paths below to point at your own data files and desired output folder. All subsequent cells reference these variables, so you only need to change them here.

In [ ]:
# =====================================================================
# USER CONFIGURATION — Set your file paths here
# =====================================================================
BIOPAC_FILE_PATH = r"sample_data/BIOPAC_04020062_9_Female_1st.txt"
BELT_FILE_PATH   = r"sample_data/BABY_BELT_04020062_9_Female_1st.csv"
OUTPUT_DIR       = r"outputs/NB01"
# =====================================================================

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {os.path.abspath(OUTPUT_DIR)}")

## 3. BIOPAC File Format (.txt, AcqKnowledge Export)

BIOPAC exports from AcqKnowledge are **tab-delimited text files** with a fixed header block followed by columnar data.

### Header Structure (lines 1–27)

| Line(s) | Content |
|---|---|
| 1 | Original `.acq` filename |
| 2 | Graph creation timestamp |
| 3 | Recording start timestamp |
| 4 | Sampling interval (`1 msec/sample` → **1 000 Hz**) |
| 5 | Number of channels |
| 6–27 | Channel names and units (pairs of lines) |

### Data Section (line 28 onward)

| Line | Content |
|---|---|
| 28 | Column headers: `milliSec  CH1  CH2  …  CH47` |
| 29 | Sample-count row (one count per channel) — **skip this** |
| 30+ | Numeric data rows (tab-delimited floats) |

### Channel Mapping

| Column | Signal | Unit | Description |
|---|---|---|---|
| `milliSec` | Time | ms | Starts at **−500 ms** (pre-trigger); trigger at 0 ms |
| `CH1` | Sync / auxiliary | Volts | **Not the ECG trace** — use for timing/sync only |
| `CH2` | PPG | mV | Photoplethysmography |
| `CH3` | ECG2 | Volts | Secondary ECG lead |
| `CH40` | **ECG (use this)** | mV | AcqKnowledge 0.05–100 Hz AHA — primary ECG for analysis |
| `CH41` | Heart Rate | BPM | Derived from R-peaks |
| `CH42` | R-R Interval | Seconds | Inter-beat interval |
| `CH43` | R-wave marker | mV | R-peak indicator pulse |
| `CH44–CH47` | Derived | BPM | Respiration rate, pulse rate, etc. |

There are **120 001 samples** in total (500 ms pre-trigger + 120 s recording at 1 kHz, with the trigger sample itself accounting for the extra count).

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)


def parse_biopac(filepath):
    """Parse BIOPAC AcqKnowledge exported .txt file.

    The file has a 27-line header, a column-name row (line 28),
    a sample-count row (line 29, skipped), then data from line 30
    onward.  We read from line 30 (skiprows=29) to avoid both the
    header and the sample-count row.

    Parameters
    ----------
    filepath : str
        Path to the BIOPAC .txt export file.

    Returns
    -------
    df : pd.DataFrame
        Parsed data with columns milliSec, CH1–CH47, and a derived
        ``time_s`` column (milliSec / 1000).
    meta : dict
        Metadata including ``filename``, ``recording_time``, and ``fs``.
    """
    meta = {}
    with open(filepath, "r") as f:
        lines = f.readlines()

    meta["filename"] = lines[0].strip()
    meta["recording_time"] = lines[2].strip().replace("Recording on: ", "")
    meta["fs"] = 1000  # 1 msec/sample

    col_names = [
        "milliSec", "CH1", "CH2", "CH3", "CH40", "CH41", "CH42",
        "CH43", "CH44", "CH45", "CH46", "CH47", "_extra",
    ]
    df = pd.read_csv(
        filepath,
        sep="\t",
        skiprows=29,
        header=None,
        names=col_names,
        on_bad_lines="skip",
    )
    df = df.drop(columns=["_extra"], errors="ignore")
    df = df.dropna(subset=["CH40"])
    df["time_s"] = df["milliSec"] / 1000.0
    return df, meta


biopac_df, biopac_meta = parse_biopac(BIOPAC_FILE_PATH)

print("=== BIOPAC Metadata ===")
for k, v in biopac_meta.items():
    print(f"  {k:20s}: {v}")

print(f"\nDataFrame shape: {biopac_df.shape}")
print(f"Time range     : {biopac_df['time_s'].iloc[0]:.3f} s  →  {biopac_df['time_s'].iloc[-1]:.3f} s")
print(f"\nFirst 10 rows:")
biopac_df.head(10)

## 4. Baby Belt File Format (.csv)

The Baby Belt device streams sensor packets over **Bluetooth Low Energy (BLE)** to a companion application that logs them as a standard CSV.

### Column Reference

| Column | Description |
|---|---|
| `PC_Time` | Host-side reception timestamp (seconds, relative) |
| `Seq` | Packet sequence number — gaps indicate dropped packets |
| `Tx_ms` | **Device-side** transmit timestamp (ms, monotonic) — more reliable than `PC_Time` |
| `Rx_ms` | Host-side reception timestamp (ms) |
| `Latency` | `Rx_ms − Tx_ms` (BLE transmission latency) |
| `InterArrival` | Time between consecutive received packets (ms) |
| `ECG` | Raw ADC counts (large integers, **not millivolts**) |
| `Resp0`, `Resp1` | Impedance-based respiration channels |
| `SpO2` | Pulse oximetry |
| `HR` | On-device heart rate estimate (BPM) |
| `IR`, `Red` | Raw optical sensor counts |
| `Temp` | Skin temperature (°F) |
| `AccX/Y/Z` | 3-axis accelerometer |
| `GyroX/Y/Z` | 3-axis gyroscope |
| `MagX/Y/Z` | 3-axis magnetometer |

### Important Considerations

1. **Irregular sampling**: The nominal rate is 100 Hz (10 ms inter-sample), but BLE scheduling jitter produces non-uniform intervals. We use `Tx_ms` (device clock) for timing and then **resample onto a uniform 100 Hz grid** via linear interpolation.
2. **Packet drops**: When `Seq` increments by more than 1, one or more packets were lost in transit.  We report these gaps for quality monitoring.
3. **ADC units**: The ECG values are raw ADC counts — they are *not* calibrated to millivolts. Their absolute amplitude cannot be compared directly with the BIOPAC signal; for cross-system comparison we normalise each trace independently.

In [ ]:
def parse_belt(filepath):
    """Parse Baby Belt textile ECG CSV file.

    Reads the CSV, constructs a time axis from ``Tx_ms`` (device-side
    timestamps), and resamples the ECG channel onto a uniform 100 Hz
    grid using linear interpolation to compensate for BLE jitter.

    Parameters
    ----------
    filepath : str
        Path to the belt CSV file.

    Returns
    -------
    df : pd.DataFrame
        Raw DataFrame exactly as read from the CSV, with an added
        ``time_s`` column and a boolean ``seq_gap`` column.
    ecg_interp : np.ndarray
        ECG resampled to a uniform 100 Hz grid.
    t_uniform : np.ndarray
        Uniform time axis in seconds corresponding to ``ecg_interp``.
    fs_nominal : int
        Nominal sampling rate (100 Hz).
    """
    df = pd.read_csv(filepath)
    df["time_s"] = (df["Tx_ms"] - df["Tx_ms"].iloc[0]) / 1000.0
    df["seq_gap"] = df["Seq"].diff() > 1

    fs_nominal = 100
    t_uniform = np.arange(0, df["time_s"].iloc[-1], 1 / fs_nominal)
    ecg_interp = np.interp(t_uniform, df["time_s"].values, df["ECG"].values)

    return df, ecg_interp, t_uniform, fs_nominal


belt_df, belt_ecg, belt_t, belt_fs = parse_belt(BELT_FILE_PATH)

print("=== Baby Belt Summary ===")
print(f"  Raw samples        : {len(belt_df)}")
print(f"  Resampled length   : {len(belt_ecg)} ({belt_fs} Hz uniform grid)")
print(f"  Duration           : {belt_t[-1]:.2f} s  ({belt_t[-1]/60:.1f} min)")
print(f"  Seq gaps (dropped) : {belt_df['seq_gap'].sum()}")
print(f"\nInterArrival statistics (ms):")
print(belt_df["InterArrival"].describe().to_string())
print(f"\nFirst 10 rows:")
belt_df.head(10)

## 5. Signal Quality Overview — Raw Waveform Comparison

Before any filtering or processing, it is valuable to inspect the raw signals side by side. We plot the first 30 seconds of:

1. **BIOPAC CH1** — sync/auxiliary channel (Volts); **do not treat as ECG**.
2. **BIOPAC CH40** — the ECG waveform (mV) after AcqKnowledge's 0.05–100 Hz AHA filter — **default channel for all ECG processing**.
3. **Baby Belt ECG** — raw ADC counts, z-score normalised for visual comparison.

Key things to look for:
- **QRS complex visibility**: How sharp and well-defined are the R-peaks in each trace?
- **Baseline wander**: Low-frequency drift caused by respiration or electrode impedance changes.
- **High-frequency noise**: Powerline interference (50/60 Hz), EMG contamination.
- **Pre-trigger region**: BIOPAC records 500 ms before the trigger (t < 0). The vertical dashed line marks the trigger onset.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

WINDOW_SEC = 30

bp_win = biopac_df[biopac_df["time_s"].between(-0.5, WINDOW_SEC)]
belt_win_mask = belt_t <= WINDOW_SEC

belt_ecg_norm = (belt_ecg - np.mean(belt_ecg)) / np.std(belt_ecg)

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(bp_win["time_s"], bp_win["CH1"], linewidth=0.4, color="#2176AE")
axes[0].axvline(0, color="red", linestyle="--", linewidth=1, label="Trigger (t=0)")
axes[0].set_ylabel("Amplitude (V)")
axes[0].set_title("BIOPAC CH1 — sync / auxiliary (not ECG)")
axes[0].legend(loc="upper right")

axes[1].plot(bp_win["time_s"], bp_win["CH40"], linewidth=0.4, color="#57A773")
axes[1].axvline(0, color="red", linestyle="--", linewidth=1, label="Trigger (t=0)")
axes[1].set_ylabel("Amplitude (mV)")
axes[1].set_title("BIOPAC CH40 — Filtered ECG (0.05–100 Hz AHA)")
axes[1].legend(loc="upper right")

axes[2].plot(belt_t[belt_win_mask], belt_ecg_norm[belt_win_mask],
             linewidth=0.4, color="#D36135")
axes[2].set_ylabel("Amplitude (z-score)")
axes[2].set_title("Baby Belt — Raw ECG (normalised ADC counts)")
axes[2].set_xlabel("Time (s)")

for ax in axes:
    ax.set_xlim(-0.5, WINDOW_SEC)

fig.suptitle("Raw Signal Comparison — First 30 Seconds", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot1_raw_signal_comparison.png"), dpi=150,
            bbox_inches="tight")
plt.show()

## 6. Synchronization

The two recording systems might not start on exact same time and typically run on different clocks, so we need a strategy to align them in time.

For this first notebook we apply a **simple duration-based trim**: we discard the BIOPAC pre-trigger region (t < 0) and then truncate both signals to their shared duration (whichever recording is shorter).  This gives us a common time window for side-by-side visualisation.

In later notebooks (NB02–NB06) we will refine alignment using **cross-correlation of detected R-peak trains**, which is far more precise but requires R-peak detection first.

In [ ]:
def synchronize_signals(biopac_df, belt_ecg, belt_t, biopac_fs=1000, belt_fs=100):
    """Synchronize BIOPAC and Belt signals by trimming to common duration.

    The BIOPAC pre-trigger region (time_s < 0) is discarded, then both
    signals are clipped to the shorter of the two durations so that
    they share an identical time span.
    CH1 is not ECG, it is sync I think. CH40 seems like ECG

    Parameters
    ----------
    biopac_df : pd.DataFrame
        BIOPAC DataFrame with ``time_s`` and ``CH40`` columns.
    belt_ecg : np.ndarray
        Resampled belt ECG array.
    belt_t : np.ndarray
        Belt time axis (seconds).
    biopac_fs : int
        BIOPAC sampling rate (default 1000 Hz).
    belt_fs : int
        Belt sampling rate (default 100 Hz).

    Returns
    -------
    biopac_ecg : np.ndarray
        Trimmed BIOPAC ECG values.
    biopac_t : np.ndarray
        Trimmed BIOPAC time axis.
    belt_ecg_trimmed : np.ndarray
        Trimmed belt ECG values.
    belt_t_trimmed : np.ndarray
        Trimmed belt time axis.
    """
        # ── 1. Strip BIOPAC pre-trigger ──────────────────────────────────────────
    biopac_post = biopac_df[biopac_df["time_s"] >= 0].copy().reset_index(drop=True)
    biopac_ecg  = biopac_post["CH40"].values
    biopac_t    = biopac_post["time_s"].values

    # ── 2. Resample belt to BIOPAC fs for cross-correlation ──────────────────
    # Use resample_poly to avoid large intermediate arrays
    g = gcd(int(biopac_fs), int(belt_fs))
    up, down = int(biopac_fs) // g, int(belt_fs) // g
    belt_ecg_rs = resample_poly(belt_ecg, up, down)

    # Align lengths to the shorter signal before correlating
    min_len = min(len(biopac_ecg), len(belt_ecg_rs))
    sig_a = biopac_ecg[:min_len]
    sig_b = belt_ecg_rs[:min_len]

    # ── 3. Cross-correlate to find lag ───────────────────────────────────────
    # Normalise both signals first so amplitude differences don't bias xcorr
    def _norm(x):
        x = x - x.mean()
        s = x.std()
        return x / s if s > 0 else x

    xcorr = correlate(_norm(sig_a), _norm(sig_b), mode="full")
    lags  = np.arange(-(min_len - 1), min_len)          # samples @ biopac_fs
    lag_samples = lags[np.argmax(xcorr)]                 # +ve → belt is late
    lag_seconds = lag_samples / biopac_fs

    print(f"  Detected lag: {lag_seconds:.4f} s  "
          f"({lag_samples} samples @ {biopac_fs} Hz)  "
          f"[+ve = belt starts later than BIOPAC]")

    # ── 4. Shift belt time axis to align with BIOPAC ─────────────────────────
    belt_t_shifted = belt_t - lag_seconds   # compensate for the offset

    # ── 5. Clip to the overlapping window ────────────────────────────────────
    t_start = max(biopac_t[0],  belt_t_shifted[0])
    t_end   = min(biopac_t[-1], belt_t_shifted[-1])

    biopac_mask = (biopac_t       >= t_start) & (biopac_t       <= t_end)
    belt_mask   = (belt_t_shifted >= t_start) & (belt_t_shifted <= t_end)

    # Re-zero both axes so they both start at 0
    t0 = t_start
    return (
        biopac_ecg[biopac_mask],  biopac_t[biopac_mask] - t0,
        belt_ecg[belt_mask],      belt_t_shifted[belt_mask] - t0,
    )



bp_ecg, bp_t, bl_ecg, bl_t = synchronize_signals(
    biopac_df, belt_ecg, belt_t,
    biopac_fs=biopac_meta["fs"], belt_fs=belt_fs,
)

print("=== Synchronized Durations ===")
print(f"  BIOPAC : {bp_t[0]:.3f} – {bp_t[-1]:.3f} s  ({len(bp_t)} samples @ {biopac_meta['fs']} Hz)")
print(f"  Belt   : {bl_t[0]:.3f} – {bl_t[-1]:.3f} s  ({len(bl_t)} samples @ {belt_fs} Hz)")
print(f"  Common : {min(bp_t[-1], bl_t[-1]):.3f} s  ({min(bp_t[-1], bl_t[-1])/60:.1f} min)")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Interactive Plotly version ---
T_START, T_END = 5.0, 15.0

bp_slice = (bp_t >= T_START) & (bp_t <= T_END)
bl_slice = (bl_t >= T_START) & (bl_t <= T_END)

fig_plotly = make_subplots(specs=[[{"secondary_y": True}]])

fig_plotly.add_trace(
    go.Scatter(x=bp_t[bp_slice], y=bp_ecg[bp_slice],
               name="BIOPAC CH40 (V)", line=dict(color="#2176AE", width=1)),
    secondary_y=False,
)
fig_plotly.add_trace(
    go.Scatter(x=bl_t[bl_slice], y=bl_ecg[bl_slice],
               name="Belt ECG (ADC)", line=dict(color="#D36135", width=1)),
    secondary_y=True,
)

fig_plotly.update_layout(
    title_text="Synchronized ECG Overlay — 10-Second Window (Interactive)",
    xaxis_title="Time (s)",
    template="seaborn",
    height=450,
)
fig_plotly.update_yaxes(title_text="BIOPAC (V)", secondary_y=False)
fig_plotly.update_yaxes(title_text="Belt (ADC counts)", secondary_y=True)
fig_plotly.show()

# --- Static matplotlib version for saving ---
fig2, ax1 = plt.subplots(figsize=(14, 4))
ax2 = ax1.twinx()

ln1 = ax1.plot(bp_t[bp_slice], bp_ecg[bp_slice],
               color="#2176AE", linewidth=0.8, label="BIOPAC CH40 (V)")
ln2 = ax2.plot(bl_t[bl_slice], bl_ecg[bl_slice],
               color="#D36135", linewidth=0.8, label="Belt ECG (ADC)")

ax1.set_xlabel("Time (s)")
ax1.set_ylabel("BIOPAC (V)", color="#2176AE")
ax2.set_ylabel("Belt (ADC counts)", color="#D36135")
ax1.set_title("Synchronized ECG Overlay — 10-Second Window")

lines = ln1 + ln2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc="upper right")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot2_synchronized_overlay.png"), dpi=150,
            bbox_inches="tight")
plt.show()

## 7. Power Spectral Density (PSD) Comparison

The **Power Spectral Density** shows how signal power is distributed across frequencies.  For ECG signals this tells us:

- **0.5–40 Hz (ECG band)**: Where the clinically relevant content lives — the P wave, QRS complex, and T wave all contribute here.
- **< 0.5 Hz**: Baseline wander from respiration or electrode motion.
- **50 or 60 Hz**: Powerline interference (country-dependent).
- **> 100 Hz**: High-frequency muscle noise, quantisation noise.

We use **Welch's method** (`scipy.signal.welch`), which averages periodograms over overlapping windows to produce a smooth, low-variance spectral estimate.  Both PSDs are plotted on a semi-log scale (linear frequency, logarithmic power) so that low-amplitude spectral features remain visible.

In [ ]:
from scipy.signal import welch

nperseg_bp = 4096
nperseg_bl = 1024

f_bp, psd_bp = welch(bp_ecg, fs=biopac_meta["fs"], nperseg=nperseg_bp)
f_bl, psd_bl = welch(bl_ecg, fs=belt_fs, nperseg=nperseg_bl)

fig3, ax = plt.subplots(figsize=(12, 5))

ax.semilogy(f_bp, psd_bp, linewidth=0.8, color="#2176AE", label="BIOPAC (1 kHz)")
ax.semilogy(f_bl, psd_bl, linewidth=0.8, color="#D36135", label="Belt (100 Hz)")

ax.axvspan(0.5, 40, alpha=0.10, color="green", label="ECG band (0.5–40 Hz)")
ax.axvline(50, color="grey", linestyle=":", linewidth=1, alpha=0.8, label="50 Hz powerline")
ax.axvline(60, color="grey", linestyle="--", linewidth=1, alpha=0.8, label="60 Hz powerline")

ax.set_xlim(0, 150)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Power Spectral Density")
ax.set_title("PSD Comparison — Welch Method")
ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plot3_psd_comparison.png"), dpi=150,
            bbox_inches="tight")
plt.show()

print(f"BIOPAC Nyquist: {biopac_meta['fs']/2} Hz  |  Belt Nyquist: {belt_fs/2} Hz")
print(f"BIOPAC PSD resolution: {f_bp[1]-f_bp[0]:.3f} Hz  |  Belt PSD resolution: {f_bl[1]-f_bl[0]:.3f} Hz")

## 8. Summary & Next Steps

### What We Accomplished

In this notebook we:

1. **Parsed** two very different ECG file formats — a BIOPAC AcqKnowledge tab-delimited export (1 kHz, 11 channels, header metadata) and a Baby Belt BLE-transmitted CSV (irregular ~100 Hz, raw ADC counts).
2. **Resampled** the belt signal onto a uniform 100 Hz grid to compensate for Bluetooth jitter.
3. **Visualised** the raw waveforms (30-second window) to get a qualitative feel for signal quality, noise characteristics, and QRS visibility in each system.
4. **Synchronised** the two recordings by trimming to their common duration.
5. **Compared power spectral densities** using Welch's method, identifying the ECG band content, potential powerline interference, and the bandwidth limitations of the lower-rate belt system.

### Key Observations

- The BIOPAC signal has 10× the sampling rate (1 kHz vs 100 Hz) and correspondingly sharper QRS morphology.
- The belt's Nyquist frequency is only 50 Hz, meaning any content above that is aliased or lost — but the core ECG band (0.5–40 Hz) is still within reach.
- Baseline wander and high-frequency noise are visible in both systems; these will be addressed by filtering in the next notebook.

### Next: Notebook 2 — Classical Filtering & R-Peak Detection

In NB02 we will:
- Apply bandpass filters to isolate the ECG band.
- Implement the Pan-Tompkins algorithm for R-peak detection.
- Compare classical peak detectors (Hamilton, Christov, NeuroKit2) across both systems.
- Quantify detection accuracy with sensitivity, PPV, and F1 score.